# Notebook 1 - Data Ingestion
### Not Yet Priced In: SEC 8-K Filing Pipeline

This notebook:
1. Loads 50 companies (ticker to CIK mapping)
2. Pulls 8-K filing metadata from SEC EDGAR for full year 2023
3. Downloads raw filing text and saves each as a local .txt file
4. Saves all filing metadata to filings_raw.csv

> Stock prices are handled separately.

---

In [ ]:
# Install dependencies (run once)
# !pip install requests pandas tqdm beautifulsoup4 lxml

In [ ]:
import requests
import pandas as pd
import os
import time
from bs4 import BeautifulSoup
from tqdm import tqdm

START_DATE  = "2023-01-01"
END_DATE    = "2023-12-31"

DATA_DIR    = "../data"
FILINGS_DIR = os.path.join(DATA_DIR, "filings_text")
FILINGS_CSV = os.path.join(DATA_DIR, "filings_raw.csv")

os.makedirs(FILINGS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

HEADERS = {
    "User-Agent": "your-name your-email@example.com",  # REQUIRED by SEC EDGAR
    "Accept-Encoding": "gzip, deflate"
}

print("Config loaded.")
print(f"  Date range  : {START_DATE} to {END_DATE}")
print(f"  Filings CSV : {FILINGS_CSV}")
print(f"  Text files  : {FILINGS_DIR}/")

Config loaded.
  Date range  : 2023-01-01 to 2023-12-31
  Filings CSV : ../data/filings_raw.csv
  Text files  : ../data/filings_text/


In [3]:
COMPANIES = {
    # Technology
    "AAPL":  "0000320193",
    "MSFT":  "0000789019",
    "GOOGL": "0001652044",
    "AMZN":  "0001018724",
    "META":  "0001326801",
    "NVDA":  "0001045810",
    "ADBE":  "0000796343",
    "CRM":   "0001108524",
    # Finance
    "JPM":   "0000019617",
    "BAC":   "0000070858",
    "GS":    "0000886982",
    "MS":    "0000895421",
    "WFC":   "0000072971",
    "BLK":   "0001364742",
    # Healthcare
    "JNJ":   "0000200406",
    "PFE":   "0000078003",
    "MRK":   "0000310158",
    "ABBV":  "0001551152",
    "UNH":   "0000731766",
    "LLY":   "0000059478",
    # Consumer / Retail
    "WMT":   "0000104169",
    "COST":  "0000909832",
    "KO":    "0000021344",
    "PEP":   "0000077476",
    "NKE":   "0000320187",
    "SBUX":  "0000829224",
    # Industrials
    "BA":    "0000012927",
    "GE":    "0000040987",
    "HON":   "0000773840",
    "CAT":   "0000018230",
    "UNP":   "0000100885",
    # Energy
    "XOM":   "0000034088",
    "CVX":   "0000093410",
    "COP":   "0001163165",
    "SLB":   "0000087347",
    # Communication / Media
    "DIS":   "0001001039",
    "NFLX":  "0001065280",
    "CMCSA": "0001166691",
    "T":     "0000732717",
    "VZ":    "0000732712",
    # Materials / Others
    "MMM":   "0000066740",
    "DOW":   "0001751788",
    "DD":    "0001666700",
    "FCX":   "0000831259",
    # Extra
    "TSLA":  "0001318605",
    "INTC":  "0000050863",
    "ORCL":  "0001341439",
    "QCOM":  "0000804328",
    "AXP":   "0000004962",
    "CVS":   "0000064803",
}

print(f"Loaded {len(COMPANIES)} companies.")
pd.DataFrame(list(COMPANIES.items()), columns=["Ticker", "CIK"])

Loaded 50 companies.


,Ticker,CIK
0,AAPL,0000320193
1,MSFT,0000789019
2,GOOGL,0001652044
3,AMZN,0001018724
4,META,0001326801
5,NVDA,0001045810
6,ADBE,0000796343
7,CRM,0001108524
8,JPM,0000019617
9,BAC,0000070858


In [13]:
import warnings
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning

# This suppresses the annoying XML warning for the rest of your session
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

def get_filings_for_company(ticker, cik, start, end):
    """
    Pulls 8-K filing metadata from SEC EDGAR.
    Uses 'primaryDocument' metadata for 100% reliable URLs.
    """
    url = f"https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json"
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        print(f"  ERROR {ticker}: {e}")
        return []

    recent       = data.get("filings", {}).get("recent", {})
    forms        = recent.get("form", [])
    dates        = recent.get("filingDate", [])
    accessions   = recent.get("accessionNumber", [])
    primary_docs = recent.get("primaryDocument", [])
    company      = data.get("name", "")

    results = []
    for form, date, acc, doc in zip(forms, dates, accessions, primary_docs):
        if form != "8-K" or not (start <= date <= end):
            continue

        acc_clean = acc.replace("-", "")
        cik_int   = int(cik)
        doc_url   = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_clean}/{doc}"

        results.append({
            "ticker":       ticker,
            "cik":          cik,
            "accession_no": acc,
            "filed_date":   date,
            "form_type":    "8-K",
            "company_name": company,
            "filing_url":   doc_url
        })
    return results

def fetch_and_save_text(f):
    """
    Downloads and saves the filing text.
    """
    ticker = f['ticker']
    acc    = f['accession_no']
    url    = f['filing_url']

    safe_acc = acc.replace("-", "_")
    filepath = os.path.join(FILINGS_DIR, f"{ticker}_{safe_acc}.txt")

    if os.path.exists(filepath):
        return filepath, "[already downloaded]"

    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        
        # Using "lxml" is fine, the warning is now suppressed upstairs
        soup = BeautifulSoup(r.text, "lxml")
        text = soup.get_text(separator=" ", strip=True)[:20000]
        
        with open(filepath, "w", encoding="utf-8") as f_out:
            f_out.write(text)
        return filepath, url
    except Exception as e:
        print(f"  WARN {ticker} {acc}: {e}")
        return None, None

print("Ingestion logic fixed and ready (Clean output).")


Ingestion logic fixed and ready (Clean output).


In [14]:
# ── Quick test on one filing before running all 50 ──────────
# Tests ABBV (the company that triggered the original error)
test_ticker = "ABBV"
test_cik    = COMPANIES[test_ticker]

print(f"Testing {test_ticker}...")
test_filings = get_filings_for_company(test_ticker, test_cik, START_DATE, END_DATE)

if test_filings:
    f = test_filings[0]
    print(f"  Found {len(test_filings)} filings.")
    print(f"  Accession : {f['accession_no']}")
    print(f"  Doc URL   : {f['filing_url']}")
    
    path, url = fetch_and_save_text(f)
    if path:
        print(f"  SUCCESS   : Saved to {path}")


Testing ABBV...
  Found 15 filings.
  Accession : 0001104659-23-124047
  Doc URL   : https://www.sec.gov/Archives/edgar/data/1551152/000110465923124047/tm2332303d1_8k.htm
  SUCCESS   : Saved to ../data/filings_text/ABBV_0001104659_23_124047.txt


In [16]:
# ── Run ingestion for all 50 companies ──────────────────────
all_filings = []
failed      = []

for ticker, cik in tqdm(COMPANIES.items(), desc="Ingesting 8-K filings"):

    # 1. Pull metadata for this company
    filings = get_filings_for_company(ticker, cik, START_DATE, END_DATE)

    # 2. Process each 8-K found
    for f in filings:
        time.sleep(0.12)  # Avoid EDGAR rate limits (10 req/sec)

        # FIX: Just pass 'f' directly now
        filepath, doc_url = fetch_and_save_text(f)

        f["local_text_path"] = filepath or ""
        f["fetch_ok"]        = filepath is not None
        all_filings.append(f)

        if not filepath:
            failed.append(f"{ticker} / {f['accession_no']}")

    time.sleep(0.1)

print(f"\nTotal filings processed : {len(all_filings)}")
print(f"Successfully saved      : {sum(f['fetch_ok'] for f in all_filings)}")
print(f"Failed / skipped        : {len(failed)}")


Ingesting 8-K filings: 100%|██████████| 50/50 [03:22<00:00,  4.04s/it]


Total filings processed : 474
Successfully saved      : 474
Failed / skipped        : 0


In [17]:
# ── Save metadata to CSV ─────────────────────────────────────
df_filings = pd.DataFrame(all_filings)[[
    "ticker", "cik", "accession_no", "filed_date",
    "form_type", "company_name", "filing_url",
    "local_text_path", "fetch_ok"
]]

df_filings.to_csv(FILINGS_CSV, index=False)

print(f"Saved : {FILINGS_CSV}")
print(f"Rows  : {len(df_filings)}")
print()
df_filings.groupby("ticker").size().reset_index(
    name="filing_count"
).sort_values("filing_count", ascending=False)

Saved : ../data/filings_raw.csv
Rows  : 474



,ticker,filing_count
11,CVS,25
4,AXP,24
23,MMM,20
20,JNJ,18
35,T,18
41,XOM,17
7,CMCSA,16
1,ABBV,15
31,PFE,14
37,UNH,14


In [18]:
# ── Final summary ────────────────────────────────────────────
csv_kb = os.path.getsize(FILINGS_CSV) / 1024
n_txt  = len([f for f in os.listdir(FILINGS_DIR) if f.endswith(".txt")])
txt_mb = sum(
    os.path.getsize(os.path.join(FILINGS_DIR, f))
    for f in os.listdir(FILINGS_DIR)
) / (1024 * 1024)

ok_count   = df_filings["fetch_ok"].sum()
fail_count = (~df_filings["fetch_ok"]).sum()

print("=" * 50)
print("NB1 COMPLETE")
print("=" * 50)
print(f"  filings_raw.csv  : {len(df_filings)} rows  ({csv_kb:.1f} KB)")
print(f"  .txt files saved : {n_txt}  ({txt_mb:.1f} MB)")
print(f"  Fetch success    : {ok_count}")
print(f"  Fetch failed     : {fail_count}")
print("=" * 50)
print()
print("Commit to GitHub:")
print("  data/filings_raw.csv")
print("  data/filings_text/*.txt")
print()
print("Run Notebook 2 next.")

NB1 COMPLETE
  filings_raw.csv  : 474 rows  (94.8 KB)
  .txt files saved : 474  (2.7 MB)
  Fetch success    : 474
  Fetch failed     : 0

Commit to GitHub:
  data/filings_raw.csv
  data/filings_text/*.txt

Run Notebook 2 next.
